# emg2pose in braindecode — embedded eegdash-viewer

Load a BIDS-converted emg2pose recording with
`braindecode.datasets.EMG2Pose` and preview it with the **eegdash-viewer**
trace viewer embedded directly in this notebook cell — including the
**synchronized hand-pose panel** (press <kbd>p</kbd> inside the viewer to
toggle, <kbd>m</kbd> cycles skeleton/mesh when a mesh sidecar exists).

This demo synthesizes one tiny recording locally; with the real release:

```bash
python scripts/export_emg2pose_bids.py --src ~/data/emg2pose_dataset_mini --out ~/data/emg2pose-bids
```

In [1]:
from pathlib import Path
import json, tempfile
import mne, numpy as np

root = Path(tempfile.mkdtemp()) / "emg2pose-bids"
ch = root / "sub-893" / "ses-s1" / "emg"
ch.mkdir(parents=True)

sfreq, dur = 1000, 30
t = np.arange(dur * sfreq) / sfreq
rhythm = 0.5 * (1 + np.sin(2 * np.pi * t / 3))
emg = (0.4 * rhythm * np.sin(2 * np.pi * 45 * t) + 0.08 * np.random.randn(dur * sfreq))
data = np.concatenate([np.tile(emg, (16, 1)), np.tile(0.6 * np.sin(2 * np.pi * t / 3), (20, 1))]) * 1e-6

info = mne.create_info([f"emg{i}" for i in range(16)] + [f"ja{i}" for i in range(20)],
                       sfreq, ["emg"] * 16 + ["misc"] * 20)
mne.export.export_raw(ch / "sub-893_ses-s1_task-wave-right_emg.vhdr",
                      mne.io.RawArray(data, info, verbose="ERROR"),
                      fmt="brainvision", overwrite=True, verbose="ERROR")
(ch / "sub-893_ses-s1_task-wave-right_emg.json").write_text(json.dumps(
    {"stage": "wave", "side": "right", "split": "train", "moving_hand": "right"}))
(root / "participants.tsv").write_text("participant_id\thandedness\nsub-893\tR\n")
print("BIDS tree ready at", root)

/Users/bruaristimunha/miniforge3/lib/python3.12/site-packages/numba/core/errors.py:193: UserWarning: Insufficiently recent colorama version found. Numba requires colorama >= 0.3.9
  warnings.warn(msg)


BIDS tree ready at /var/folders/jw/89j4npnx5vb2vcrknd6d8y740000gn/T/tmpq0polfjx/emg2pose-bids


/Users/bruaristimunha/miniforge3/lib/python3.12/site-packages/pybv/io.py:682: UserWarning: Encountered unsupported non-voltage units: n/a
Note that the BrainVision format specification supports only µV.
  warn(msg)


## Load with generalized metadata fields

Every sidecar/participants column flows into `records` verbatim —
`stage`, `side`, `split`, `handedness`, ... with no dataset-specific code.

In [2]:
from braindecode.datasets import EMG2Pose

dataset = EMG2Pose(root)
dataset.records[["subject", "session", "task", "stage", "side", "split", "handedness"]]

,subject,session,task,stage,side,split,handedness
0,893,s1,wave-right,wave,right,train,R


## Interactive viewer

`plot()` serves the vendored viewer + this recording over a localhost
server and embeds it below. Drag to pan, hover for the cursor readout —
the hand skeleton tracks your cursor.

In [3]:
dataset.plot(index=0)

/Users/bruaristimunha/miniforge3/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
